# Transverse field Ising model

$$
H = -J\sum_{i=1}^{N-1} \sigma_i^z \sigma_{i+1}^z
    - h\sum_{i=1}^{N} \sigma_i^x,
\qquad
J,\;h \in \mathbb{R},
\qquad
J > 0,\; h \geq 0
$$

The full Hilbert space $\mathcal{H}$ is the product of spin-$1/2$ spaces, and $\mathcal{H}_{1/2} \cong \mathbb{C}^2$, so $\mathcal{H} = \otimes_{i=1}^N (\mathbb{C}^2)$. 
Thus, the first term is properly a tensor product,

$$
\sigma_i^z \sigma_{i+1}^z
=
I \otimes \cdots \otimes
\underbrace{\sigma^z}_{i\text{-th site}}
\otimes
\underbrace{\sigma^z}_{{(i+1)}\text{-th site}}
\otimes \cdots \otimes I,
$$

and likewise, $\sigma_x^i$ acts on the $i$-th site.

In [43]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True) # Enable 64-bit dtypes

# Define Pauli matrices
sigma_x = jnp.array([[0, 1], 
                     [1, 0]], dtype=jnp.float64)
sigma_z = jnp.array([[1, 0], 
                     [0, -1]], dtype=jnp.float64)

I = jnp.eye(2, dtype=jnp.float64)

The matrices act on the factored $\mathcal{H}_{1/2}$ spaces. Define the operator on the full Hilbert space.

In [ ]:
def act_on_site(operator, site, num_sites):
    result = 1

    for i in range(num_sites):
        if i == site:
            result = jnp.kron(result, operator) # jnp.kron is the Kronecker product, i.e. tensor product/element-wise multiplication
        else:
            result = jnp.kron(result, I)
    return result

Define the Hamiltonian

In [45]:
def tf_ising(N, J, h):
    dim = 2 ** N
    H = jnp.zeros((dim, dim), dtype=jnp.float64) # Empty 2^n square matrix

    # Spin coupling terms
    for i in range(N-1):
        H -= J * act_on_site(sigma_z, i, N) @ act_on_site(sigma_z, i+1, N)

    # Transverse field terms
    for i in range(N):
        H -= h * act_on_site(sigma_x, i, N)

    return H

Diagonalize the matrix using usual **numpy** `linalg.eigh()` routine, returns ***normalized*** eigenvectors/values. 

i.e. $E = [E_0, E_1, \cdots],\; V = [|\psi_0 \rangle, |\psi_1 \rangle, \cdots]$

In [46]:
N = 3
J = 1.0
h = 0.7

H = tf_ising(N, J, h)

E, V = jnp.linalg.eigh(H)

Display the ground state and eigenvalue in $\LaTeX$

In [ ]:
from IPython.display import Math, display

def display_ground_state(E, V, N):
    state = jax.device_get(V[:, 0]) 
    state = state * (1 if state[abs(state).argmax()] >= 0 else -1) # Global phase, for positive coefficients
    terms = []
    for i, amplitude in enumerate(state):
        bin = f"{i:0{N}b}" # Binary representation of index i with N bits, leading zeroes
        ket = "".join(r"\uparrow" if bit == "0" else r"\downarrow" for bit in bin)
        terms.append(rf"{amplitude:.6f}\lvert {ket}\rangle")
    expansion = " + ".join(terms).replace("+ -", "- ") # Join string and tidy signs
    display(Math(rf"E_0 = {float(E[0]):.10f}"))
    display(Math(rf"\lvert\psi_0\rangle \approx {expansion}"))

display_ground_state(E, V, N)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

The average $z$-magnetization operator is $M_z = \frac{1}{N} \sum_{i} \sigma_i^z$. The average $z$-magnetization in the ground state is the expectation value,

$$
 \langle \psi_0 | M_z | \psi_0 \rangle.
$$

In [ ]:
Mz = jnp.zeros_like(H)

for i in range(N):
    Mz += act_on_site(sigma_z, i, N)

Mz /= N

psi0 = jax.device_get(V[:, 0]) # 0th column of V (eigenvector matrix)

mz = jnp.vdot(psi0, Mz @ psi0).real # Mz @ psi0 is a matrix-vector product, etc.

display(Math(rf"\langle M_z \rangle = {mz:.16f}"))

1.108630014587062e-15


<IPython.core.display.Math object>

Zero longitudinal magnetization, as expected, since the Hamiltonian has global symmetry under flipping every spin; recall the longitudinal part 
$$
\mathcal{H}_{\text{long}} = -J\sum_i\sigma_z^i\sigma_z^{i+1}, \\
\sigma_i^z \rightarrow -\sigma_i^z \iff \mathcal{H}_{\text{long}} \rightarrow \mathcal{H}_{\text{long}} 
$$
and obviously the transverse part is unchanged. However under parity transformation of $\sigma_x$ there is no such symmetry for the transverse part. So we compute the same quantity for the $M_x$ operator, defined exactly like $M_z, \langle M_z \rangle$. 

In [65]:
Mx = jnp.zeros_like(H)

for i in range(N):
    Mx += act_on_site(sigma_x, i, N)

Mx /= N

mx = jnp.vdot(psi0, Mx @ psi0).real

display(Math(rf"\langle M_x \rangle = {mx:.16f}"))

<IPython.core.display.Math object>